# PINN for Tumor Growth on TCGA-GBM

End-to-end Physics-Informed Neural Network pipeline for the 2-D
Fisher-Kolmogorov tumor-growth model, calibrated against real glioblastoma
imaging from **TCGA-GBM** (The Cancer Imaging Archive).

Data flow:

* Load co-registered, skull-stripped MRI volumes + tumor segmentation
  (NIfTI format, as distributed by the BraTS pre-processed TCGA-GBM cohort).
* Build a 2-D axial slice through the tumor centroid.
* Derive a white-/gray-matter tissue mask from T1 intensity.
* Treat the (noisy, thresholded) tumor segmentation as an observation field
  u(x,y,t) ∈ [0,1] — a density proxy used only at the data-loss term.
* Calibrate `D_wm`, `D_gm`, `ρ` by solving the inverse Fisher-KPP problem.
* If multiple longitudinal scans are supplied, each is used as an
  independent time snapshot; otherwise the tumor seed (centroid at
  inferred onset) contributes the early time point and the scan
  contributes the late one (documented identifiability caveat).

**No external data?** The notebook auto-falls back to a synthetic
Fisher-KPP ground truth so every cell still runs.

Sections:
1. Setup & Imports
2. Configuration
3. TCGA-GBM Data Loading (with synthetic fallback)
4. PINN Architecture
5. Loss Functions & Adaptive Weighting
6. Training Loop
7. Uncertainty Quantification
8. Evaluation & Metrics
9. Visualization
10. Results Summary


## 1. Setup & Imports

In [ ]:
import sys, subprocess, importlib, os, math, random, json, pathlib, glob, warnings

def _ensure(pkg, import_name=None):
    try: importlib.import_module(import_name or pkg)
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for p, i in [("numpy", None), ("scipy", None), ("matplotlib", None),
             ("torch", "torch"), ("tqdm", "tqdm"), ("pyDOE2", "pyDOE2"),
             ("nibabel", "nibabel")]:
    _ensure(p, i)

import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import splu
from scipy.ndimage import binary_erosion, center_of_mass, zoom, gaussian_filter
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from pyDOE2 import lhs
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional
import nibabel as nib

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float32
torch.set_default_dtype(DTYPE)

RESULTS_DIR = pathlib.Path("./results"); RESULTS_DIR.mkdir(exist_ok=True)

print(f"Torch {torch.__version__} | Device: {DEVICE} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Configuration

Everything is parameterized through `PINNConfig`.
`tcga_data_root` points to a folder with BraTS/TCGA-GBM NIfTI volumes
(`*_t1.nii.gz`, `*_t1ce.nii.gz`, `*_flair.nii.gz`, `*_seg.nii.gz`). If it is
unset or empty the synthetic fallback is used.

**Expected layout** (BraTS-style, from the TCGA-GBM cohort):
```
<tcga_data_root>/<CaseID>/<CaseID>_t1.nii.gz
                          <CaseID>_t1ce.nii.gz
                          <CaseID>_flair.nii.gz
                          <CaseID>_seg.nii.gz  # label map: 0 bg, 1 necrosis, 2 edema, 4 enhancing
```
For longitudinal cases you may place multiple sessions as
`<CaseID>/<session_days>/...` where `<session_days>` is the number of days
since the earliest scan (e.g. `0`, `90`, `210`).


In [ ]:
@dataclass
class PINNConfig:
    # ---- TCGA-GBM dataset --------------------------------------- #
    tcga_data_root: str = ""       # set to local path, or leave "" for synthetic
    tcga_case_id:   str = ""       # "" -> first case found under data_root
    tcga_modality_for_tissue: str = "t1"   # used to derive WM/GM mask
    tcga_slice_axis: int = 2       # 2 = axial
    tcga_slice_selection: str = "tumor_centroid"  # or "mid" or an int
    tcga_target_grid: int = 128    # resample slice to NxN for training
    tcga_physical_mm_per_vox: float = 1.0   # BraTS is 1mm iso after preprocessing
    # Days-since-onset assumed for a single-session case (heuristic):
    tcga_assumed_single_session_day: float = 200.0
    # If segmentation is binary, scale to a density proxy in [0,1]:
    seg_density_smooth_sigma: float = 1.5

    # ---- Domain (filled in by data loader) ---------------------- #
    Lx: float = 100.0
    Ly: float = 100.0
    T:  float = 250.0
    x0: float = 50.0
    y0: float = 50.0
    sigma0: float = 5.0
    gm_radius: float = 30.0
    gm_center: Tuple[float, float] = (50.0, 50.0)

    # ---- Synthetic-fallback true parameters --------------------- #
    D_wm_true: float = 0.25
    D_gm_true: float = 0.025
    rho_true:  float = 0.012

    # ---- FD fallback ------------------------------------------- #
    fd_nx: int = 101; fd_ny: int = 101
    fd_dt: float = 0.5
    fd_snapshot_times: Tuple[float, ...] = (50.0, 100.0, 200.0, 250.0)

    # ---- Observations ------------------------------------------ #
    obs_times: Tuple[float, ...] = (50.0, 100.0, 200.0)
    n_obs_total: int = 2000
    obs_noise_std: float = 0.05     # higher for real data
    T_pred: float = 250.0
    heldout_fraction: float = 0.15  # train/val split on obs points

    # ---- Network ---------------------------------------------- #
    hidden_layers: int = 6
    hidden_width: int = 128
    dropout_p: float = 0.05
    fourier_num_features: int = 32
    fourier_sigma: float = 3.0
    use_sigmoid_output: bool = True

    # ---- Priors (log-space literature values) ------------------ #
    log_D_wm_prior_mean: float = math.log(0.3)
    log_D_wm_prior_std:  float = 0.7
    log_D_gm_prior_mean: float = math.log(0.03)
    log_D_gm_prior_std:  float = 0.7
    log_rho_prior_mean:  float = math.log(0.01)
    log_rho_prior_std:   float = 0.7
    init_param_noise: float = 0.15

    # ---- Collocation -------------------------------------------- #
    N_f: int = 20_000; N_ic: int = 2_000; N_bc: int = 1_000
    rar_every: int = 500
    rar_fraction: float = 0.20
    rar_candidate_mult: int = 5
    rar_top_fraction: float = 0.20

    # ---- Losses / training ------------------------------------- #
    lambda_init: Tuple[float, ...] = (1.0, 100.0, 10.0, 100.0, 1.0, 10.0)
    adaptive_every: int = 1000
    adaptive_alpha: float = 0.9
    phase1_iters: int = 5_000; phase1_lr: float = 1e-3
    phase2_iters: int = 20_000; phase2_lr: float = 5e-4
    phase3_iters: int = 2_000
    grad_clip_norm: float = 1.0
    cosine_T0: int = 5_000; cosine_Tmult: int = 2
    curriculum_every: int = 2_000
    curriculum_start_frac: float = 0.25
    curriculum_growth: float = 1.35

    # ---- UQ ----------------------------------------------------- #
    mcd_samples: int = 200; ensemble_size: int = 5; laplace_samples: int = 1000

    # ---- Eval --------------------------------------------------- #
    dice_threshold: float = 0.16
    eval_grid: int = 101
    log_every: int = 200

    quick_mode: bool = False

    def __post_init__(self):
        if self.quick_mode:
            self.phase1_iters = 1000; self.phase2_iters = 3000
            self.phase3_iters = 300; self.N_f = 4000
            self.mcd_samples = 50; self.ensemble_size = 2
            self.laplace_samples = 200

cfg = PINNConfig(
    # >>> EDIT THIS to point at your TCGA-GBM / BraTS download <<<<<
    tcga_data_root = os.environ.get("TCGA_DATA_ROOT", ""),
    tcga_case_id   = os.environ.get("TCGA_CASE_ID", ""),
)
print(cfg)


## 3. TCGA-GBM Data Loading

### 3a. Locate sessions on disk

The loader walks `tcga_data_root/<case>/` for NIfTI volumes. If a case
directory contains subdirectories named as integers, each is treated as an
independent longitudinal session (integer = days since earliest scan).
Otherwise the case is treated as a single-session acquisition at day
`tcga_assumed_single_session_day`.


In [ ]:
def discover_tcga_sessions(root: str, case_id: str = "") -> List[Dict]:
    """Return a list of {case, day, files{t1,t1ce,flair,seg}} dicts.

    Returns [] if nothing is found — callers should then use the synthetic fallback.
    """
    if not root or not os.path.isdir(root):
        return []
    cases = [case_id] if case_id else sorted(
        [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    out = []
    for case in cases:
        case_dir = os.path.join(root, case)
        if not os.path.isdir(case_dir): continue
        subdirs = [d for d in os.listdir(case_dir)
                   if os.path.isdir(os.path.join(case_dir, d)) and d.lstrip("-").isdigit()]
        if subdirs:
            for d in sorted(subdirs, key=int):
                sess_dir = os.path.join(case_dir, d)
                files = _find_modalities(sess_dir, case)
                if files["seg"]: out.append(dict(case=case, day=float(d), files=files))
        else:
            files = _find_modalities(case_dir, case)
            if files["seg"]:
                out.append(dict(case=case, day=0.0, files=files))
        if case_id: break
    return out


def _find_modalities(folder: str, case: str) -> Dict[str, Optional[str]]:
    """Match BraTS-style suffixes _t1/_t1ce/_flair/_seg."""
    mapping = {"t1": None, "t1ce": None, "flair": None, "t2": None, "seg": None}
    for f in glob.glob(os.path.join(folder, "*.nii*")):
        name = os.path.basename(f).lower()
        for k in mapping:
            if f"_{k}." in name or name.endswith(f"_{k}.nii") or name.endswith(f"_{k}.nii.gz"):
                mapping[k] = f
    return mapping


def load_volume(path: str) -> Tuple[np.ndarray, np.ndarray]:
    """Load a NIfTI volume; return (data, affine). Data is cast to float32."""
    img = nib.load(path)
    return np.asanyarray(img.dataobj).astype(np.float32), img.affine


def tumor_density_from_seg(seg: np.ndarray, sigma: float) -> np.ndarray:
    """Map BraTS-style integer labels into a density proxy in [0, 1]:
        label 1 (necrosis) -> 1.0
        label 4 (enhancing) -> 0.9
        label 2 (edema)    -> 0.4
    Then Gaussian-smooth to obtain a continuous field."""
    d = np.zeros_like(seg, dtype=np.float32)
    d[seg == 1] = 1.0
    d[seg == 4] = 0.9
    d[seg == 2] = 0.4
    if sigma > 0: d = gaussian_filter(d, sigma=sigma)
    return np.clip(d, 0.0, 1.0)


def derive_tissue_mask(t1: np.ndarray, brain_mask: np.ndarray) -> np.ndarray:
    """Very coarse WM (1) vs GM (0) mask from T1 intensity within the brain.

    Uses Otsu-style two-class thresholding on intra-brain intensities.
    In T1w images, WM is brighter than GM — return 1 where intensity >
    threshold AND inside the brain."""
    if brain_mask.sum() == 0:
        return np.zeros_like(t1)
    vals = t1[brain_mask > 0]
    if vals.size == 0: return np.zeros_like(t1)
    # Otsu
    hist, edges = np.histogram(vals, bins=128)
    p = hist / (hist.sum() + 1e-12)
    cum = np.cumsum(p); mu = np.cumsum(p * 0.5 * (edges[:-1] + edges[1:]))
    mu_t = mu[-1]
    denom = cum * (1.0 - cum); denom[denom == 0] = 1e-12
    sigma_b2 = (mu_t * cum - mu) ** 2 / denom
    thr = 0.5 * (edges[:-1] + edges[1:])[np.nanargmax(sigma_b2)]
    mask = ((t1 > thr) & (brain_mask > 0)).astype(np.float32)
    return mask


### 3b. Preprocess: pick slice, normalize, resample

In [ ]:
def select_slice_index(seg_vol: np.ndarray, axis: int, mode) -> int:
    if isinstance(mode, int): return int(mode)
    if mode == "mid": return seg_vol.shape[axis] // 2
    # default: slice through tumor centroid along given axis
    if (seg_vol > 0).sum() == 0: return seg_vol.shape[axis] // 2
    com = center_of_mass(seg_vol > 0)
    return int(round(com[axis]))


def resample_2d(arr: np.ndarray, target: int) -> np.ndarray:
    h, w = arr.shape
    return zoom(arr, (target / h, target / w), order=1)


def build_session_frame(session: Dict, cfg: PINNConfig) -> Dict:
    """Load one session, extract the 2-D slice, compute tissue + density.

    Returns keys: day, t1, t1ce, flair, density, brain_mask, wm_mask, extent.
    """
    seg, _ = load_volume(session["files"]["seg"])
    t1_path = session["files"][cfg.tcga_modality_for_tissue] or session["files"]["t1"]
    if t1_path is None: raise RuntimeError("No T1 modality found for tissue mask")
    t1, _ = load_volume(t1_path)
    axis = cfg.tcga_slice_axis
    sl = select_slice_index(seg, axis, cfg.tcga_slice_selection)

    def take(v):
        if axis == 0: return v[sl, :, :]
        if axis == 1: return v[:, sl, :]
        return v[:, :, sl]

    t1_2d  = take(t1); seg_2d = take(seg)
    # Brain mask: anywhere inside T1 with non-zero intensity (assumes skull-stripped)
    brain = (t1_2d > (0.02 * t1_2d.max())).astype(np.float32)
    # Optional modalities
    t1ce_p = session["files"].get("t1ce"); flair_p = session["files"].get("flair")
    t1ce_2d  = take(load_volume(t1ce_p)[0])  if t1ce_p  else None
    flair_2d = take(load_volume(flair_p)[0]) if flair_p else None

    wm = derive_tissue_mask(t1_2d, brain)

    # Resample every layer to target grid
    tgt = cfg.tcga_target_grid
    t1_2d  = resample_2d(t1_2d, tgt)
    seg_2d = resample_2d(seg_2d, tgt)
    brain  = (resample_2d(brain, tgt) > 0.5).astype(np.float32)
    wm     = (resample_2d(wm, tgt) > 0.5).astype(np.float32) * brain
    if t1ce_2d  is not None: t1ce_2d  = resample_2d(t1ce_2d,  tgt)
    if flair_2d is not None: flair_2d = resample_2d(flair_2d, tgt)

    # Tumor "density" proxy in [0,1]
    density = tumor_density_from_seg(seg_2d, cfg.seg_density_smooth_sigma) * brain

    # Physical extent: BraTS after preprocessing is 1 mm/voxel isotropic.
    # After resampling to target grid, each pixel is original_dim * 1mm / tgt.
    mm_per_vox = cfg.tcga_physical_mm_per_vox * (t1.shape[0 if axis > 0 else 1] / tgt)
    Lx = density.shape[1] * mm_per_vox
    Ly = density.shape[0] * mm_per_vox

    return dict(day=session["day"], t1=t1_2d, t1ce=t1ce_2d, flair=flair_2d,
                density=density, brain=brain, wm=wm, Lx=Lx, Ly=Ly,
                case=session["case"], slice_index=sl)


sessions = discover_tcga_sessions(cfg.tcga_data_root, cfg.tcga_case_id)
print(f"Discovered {len(sessions)} TCGA-GBM session(s).")
for s in sessions:
    print(f"  case={s['case']}  day={s['day']:.0f}  seg={os.path.basename(s['files']['seg'])}")

USE_REAL_DATA = len(sessions) > 0
frames = []
if USE_REAL_DATA:
    for s in sessions: frames.append(build_session_frame(s, cfg))
    # Align day 0 to earliest session; configure domain from first frame
    t0 = min(f["day"] for f in frames)
    for f in frames: f["day"] -= t0
    if len(frames) == 1:
        # Single session: assume scan is at tcga_assumed_single_session_day,
        # and supply a synthetic "early" observation = tumor seed at day 0.
        frames[0]["day"] = cfg.tcga_assumed_single_session_day
    # Update cfg domain
    cfg.Lx = float(frames[0]["Lx"]); cfg.Ly = float(frames[0]["Ly"])
    cfg.T  = max(cfg.T, max(f["day"] for f in frames) + 20.0)
    # Seed = centroid of earliest frame's density
    ref = frames[0]
    if (ref["density"] > 0.1).sum() > 0:
        com = center_of_mass(ref["density"] > 0.1)
        cfg.y0 = float(com[0]) * cfg.Ly / ref["density"].shape[0]
        cfg.x0 = float(com[1]) * cfg.Lx / ref["density"].shape[1]
    cfg.obs_times = tuple(float(f["day"]) for f in frames)
    cfg.T_pred = float(frames[-1]["day"])
    print(f"\nDomain: {cfg.Lx:.1f} x {cfg.Ly:.1f} mm, T={cfg.T:.1f} d, "
          f"seed=({cfg.x0:.1f},{cfg.y0:.1f}), obs_times={cfg.obs_times}")
else:
    print("\n>>> No TCGA data found — falling back to synthetic Fisher-KPP ground truth. <<<")


### 3c. Tissue mask → heterogeneous D, observations, and IC

For TCGA: the WM mask from T1 gives the heterogeneous D-field; observation
points are drawn from the tumor density proxy; the IC is a Gaussian seed at
the tumor centroid. For the synthetic fallback we run the ADI Crank-Nicolson
solver from the previous version.


In [ ]:
# --------- Synthetic fallback utilities (also used for FD viz) ---- #
def tissue_mask_synth(cfg, nx, ny):
    xs = np.linspace(0, cfg.Lx, nx); ys = np.linspace(0, cfg.Ly, ny)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    is_gm = (X - cfg.gm_center[0]) ** 2 + (Y - cfg.gm_center[1]) ** 2 <= cfg.gm_radius ** 2
    return np.where(is_gm, 0.0, 1.0)

def diffusion_field_np(cfg, nx, ny, D_wm, D_gm, wm=None):
    if wm is None: wm = tissue_mask_synth(cfg, nx, ny)
    return D_wm * wm + D_gm * (1.0 - wm)

def initial_condition_np(cfg, nx, ny):
    xs = np.linspace(0, cfg.Lx, nx); ys = np.linspace(0, cfg.Ly, ny)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    return np.exp(-((X - cfg.x0) ** 2 + (Y - cfg.y0) ** 2) / (2.0 * cfg.sigma0 ** 2))

def _bilinear(xg, yg, U, xs, ys):
    nx, ny = len(xg), len(yg); dx = xg[1] - xg[0]; dy = yg[1] - yg[0]
    fi = np.clip((xs - xg[0]) / dx, 0, nx - 1 - 1e-9)
    fj = np.clip((ys - yg[0]) / dy, 0, ny - 1 - 1e-9)
    i0 = fi.astype(int); j0 = fj.astype(int); a = fi - i0; b = fj - j0
    return (U[j0, i0]*(1-a)*(1-b) + U[j0, i0+1]*a*(1-b) +
            U[j0+1, i0]*(1-a)*b + U[j0+1, i0+1]*a*b)

# Synthetic FD solver (ADI) — only used when USE_REAL_DATA is False
def _build_1d_diffusion(D_line, h):
    n = len(D_line); main = np.zeros(n); up = np.zeros(n-1); lo = np.zeros(n-1)
    for i in range(n):
        c = 0.0
        if i+1 < n:
            Df = 2*D_line[i]*D_line[i+1]/(D_line[i]+D_line[i+1]+1e-30); up[i] = Df/h**2; c -= Df/h**2
        if i-1 >= 0:
            Df = 2*D_line[i]*D_line[i-1]/(D_line[i]+D_line[i-1]+1e-30); lo[i-1] = Df/h**2; c -= Df/h**2
        main[i] = c
    return sp.diags([lo, main, up], offsets=[-1, 0, 1], format="csc")

def solve_fisher_kpp_adi(cfg, D_wm, D_gm, rho, T_end, wm_field=None):
    nx, ny = cfg.fd_nx, cfg.fd_ny
    dx = cfg.Lx/(nx-1); dy = cfg.Ly/(ny-1); dt = cfg.fd_dt
    n_steps = int(round(T_end / dt))
    D = diffusion_field_np(cfg, nx, ny, D_wm, D_gm, wm_field)
    I_x = sp.eye(nx, format="csc"); I_y = sp.eye(ny, format="csc")
    Lx_rows = [_build_1d_diffusion(D[j, :], dx) for j in range(ny)]
    Ly_cols = [_build_1d_diffusion(D[:, i], dy) for i in range(nx)]
    Ax = [splu((I_x - 0.5*dt*Lx).tocsc()) for Lx in Lx_rows]
    Bx = [I_x + 0.5*dt*Lx for Lx in Lx_rows]
    Ay = [splu((I_y - 0.5*dt*Ly).tocsc()) for Ly in Ly_cols]
    By = [I_y + 0.5*dt*Ly for Ly in Ly_cols]
    u = initial_condition_np(cfg, nx, ny)
    snaps = {0.0: u.copy()}
    for step in range(1, n_steps+1):
        us = np.empty_like(u)
        for j in range(ny): us[j,:] = Ax[j].solve(Bx[j] @ u[j,:])
        un = np.empty_like(u)
        for i in range(nx): un[:,i] = Ay[i].solve(By[i] @ us[:,i])
        un = un + dt * rho * un * (1.0 - un)
        u = np.clip(un, 0.0, 1.0)
        t_now = step * dt
        if any(abs(t_now - ts) < 0.5*dt for ts in cfg.fd_snapshot_times):
            snaps[round(t_now, 4)] = u.copy()
    return dict(snaps=snaps, x=np.linspace(0, cfg.Lx, nx),
                y=np.linspace(0, cfg.Ly, ny), D=D)

# --------- Build observations dictionary -------------------------- #
rng = np.random.default_rng(SEED)

if USE_REAL_DATA:
    # Use the loaded frames: sample observation points from each session's density.
    per = cfg.n_obs_total // len(frames)
    ox, oy, ot, ou = [], [], [], []
    for f in frames:
        H, W = f["density"].shape
        xs = rng.uniform(0, cfg.Lx, per); ys = rng.uniform(0, cfg.Ly, per)
        xi = np.linspace(0, cfg.Lx, W); yi = np.linspace(0, cfg.Ly, H)
        vals = _bilinear(xi, yi, f["density"], xs, ys)
        vals = np.clip(vals + rng.normal(0, cfg.obs_noise_std, per), 0, 1)
        ox.append(xs); oy.append(ys); ot.append(np.full_like(xs, f["day"])); ou.append(vals)
    obs = dict(x=np.concatenate(ox), y=np.concatenate(oy),
               t=np.concatenate(ot), u=np.concatenate(ou))
    # WM mask on canonical grid for the PINN's D field
    wm_full = frames[0]["wm"].astype(np.float32)
    # Resample wm to (fd_nx, fd_ny) for FD visualization
    wm_fd = resample_2d(wm_full, cfg.fd_nx)
    # Build a "reference" FD solution using the RECOVERED params later;
    # placeholder here uses literature-ish values just for the visualizer.
    fd = solve_fisher_kpp_adi(cfg, 0.25, 0.025, 0.012, cfg.T, wm_field=wm_fd)
    # No "true" snapshot for comparison -> will compare to held-out real obs.
    TRUE_U_AT_T_PRED = frames[-1]["density"]
    TRUE_U_GRID_X = np.linspace(0, cfg.Lx, TRUE_U_AT_T_PRED.shape[1])
    TRUE_U_GRID_Y = np.linspace(0, cfg.Ly, TRUE_U_AT_T_PRED.shape[0])
else:
    # Synthetic FD ground truth + noisy observations
    fd = solve_fisher_kpp_adi(cfg, cfg.D_wm_true, cfg.D_gm_true, cfg.rho_true, cfg.T)
    wm_full = tissue_mask_synth(cfg, cfg.tcga_target_grid, cfg.tcga_target_grid).astype(np.float32)
    per = cfg.n_obs_total // len(cfg.obs_times)
    ox, oy, ot, ou = [], [], [], []
    for ts in cfg.obs_times:
        key = min(fd["snaps"].keys(), key=lambda k: abs(k - ts))
        U = fd["snaps"][key]
        xs = rng.uniform(0, cfg.Lx, per); ys = rng.uniform(0, cfg.Ly, per)
        v = _bilinear(fd["x"], fd["y"], U, xs, ys)
        v = np.clip(v + rng.normal(0, cfg.obs_noise_std, per), 0, 1)
        ox.append(xs); oy.append(ys); ot.append(np.full_like(xs, key)); ou.append(v)
    obs = dict(x=np.concatenate(ox), y=np.concatenate(oy),
               t=np.concatenate(ot), u=np.concatenate(ou))
    key = min(fd["snaps"].keys(), key=lambda k: abs(k - cfg.T_pred))
    TRUE_U_AT_T_PRED = fd["snaps"][key]
    TRUE_U_GRID_X = fd["x"]; TRUE_U_GRID_Y = fd["y"]

# Held-out validation split (same for both modes)
n_total = len(obs["u"])
idx_all = rng.permutation(n_total)
n_val = int(cfg.heldout_fraction * n_total)
VAL_IDX = idx_all[:n_val]; TRAIN_IDX = idx_all[n_val:]
obs_train = {k: v[TRAIN_IDX] for k, v in obs.items()}
obs_val   = {k: v[VAL_IDX]   for k, v in obs.items()}
print(f"Observations: {len(obs['u'])} total "
      f"(train={len(obs_train['u'])}, val={len(obs_val['u'])}) across t={sorted(set(obs['t']))}")
print(f"Using {'REAL TCGA-GBM data' if USE_REAL_DATA else 'synthetic FD ground truth'}.")


## 4. PINN Architecture

Fourier-feature input embedding, Tanh-with-trainable-slope layers, dropout,
sigmoid output, log-space trainable `D_wm`, `D_gm`, `ρ`. For TCGA mode the
spatial D-field is interpolated from the **patient-specific** WM mask at
query time — so the same model form transfers straight from synthetic to
real data.


In [ ]:
class FourierFeatures(nn.Module):
    def __init__(self, in_dim, num_features, sigma, seed=0):
        super().__init__()
        g = torch.Generator().manual_seed(seed)
        self.register_buffer("B", torch.randn(in_dim, num_features, generator=g) * sigma)
    def forward(self, x):
        proj = 2 * math.pi * x @ self.B
        return torch.cat([torch.cos(proj), torch.sin(proj)], dim=-1)


class TanhSlopeLinear(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin = nn.Linear(d_in, d_out)
        nn.init.xavier_uniform_(self.lin.weight); nn.init.zeros_(self.lin.bias)
        self.alpha = nn.Parameter(torch.tensor(1.0))
    def forward(self, x): return torch.tanh(self.alpha * self.lin(x))


class TumorPINN(nn.Module):
    def __init__(self, cfg, wm_mask_np=None, seed=SEED):
        super().__init__()
        self.cfg = cfg
        torch.manual_seed(seed)
        self.ff = FourierFeatures(3, cfg.fourier_num_features, cfg.fourier_sigma, seed=seed)
        d = 2 * cfg.fourier_num_features
        layers = []
        for _ in range(cfg.hidden_layers):
            layers.append(TanhSlopeLinear(d, cfg.hidden_width))
            if cfg.dropout_p > 0: layers.append(nn.Dropout(cfg.dropout_p))
            d = cfg.hidden_width
        self.body = nn.Sequential(*layers)
        self.head = nn.Linear(d, 1)
        nn.init.xavier_uniform_(self.head.weight); nn.init.zeros_(self.head.bias)

        def _init(mu):
            return nn.Parameter(torch.tensor(
                mu + cfg.init_param_noise*(2*torch.rand(1).item()-1), dtype=DTYPE))
        self.log_D_wm = _init(cfg.log_D_wm_prior_mean)
        self.log_D_gm = _init(cfg.log_D_gm_prior_mean)
        self.log_rho  = _init(cfg.log_rho_prior_mean)

        # Patient-specific WM mask as a buffer (for TCGA); None triggers analytic disc.
        if wm_mask_np is not None:
            self.register_buffer("wm_mask", torch.tensor(wm_mask_np, dtype=DTYPE))
        else:
            self.register_buffer("wm_mask", torch.empty(0))

    def physical_params(self):
        return torch.exp(self.log_D_wm), torch.exp(self.log_D_gm), torch.exp(self.log_rho)

    def freeze_physics(self, frozen=True):
        for p in (self.log_D_wm, self.log_D_gm, self.log_rho): p.requires_grad_(not frozen)

    def network_parameters(self):
        phys = {id(self.log_D_wm), id(self.log_D_gm), id(self.log_rho)}
        return [p for p in self.parameters() if id(p) not in phys]

    def physics_parameters(self):
        return [self.log_D_wm, self.log_D_gm, self.log_rho]

    def forward(self, x, y, t):
        xn = 2*x/self.cfg.Lx - 1; yn = 2*y/self.cfg.Ly - 1; tn = 2*t/self.cfg.T - 1
        h = self.ff(torch.cat([xn, yn, tn], dim=-1))
        h = self.body(h); out = self.head(h)
        return torch.sigmoid(out) if self.cfg.use_sigmoid_output else out


class PINNEnsemble(nn.Module):
    def __init__(self, cfg, wm_mask_np, K):
        super().__init__()
        self.members = nn.ModuleList(
            [TumorPINN(cfg, wm_mask_np, seed=SEED + k*101) for k in range(K)])
    def __iter__(self): return iter(self.members)


# Print sanity check
_net_dummy = TumorPINN(cfg, wm_mask_np=wm_full).to(DEVICE)
_x = torch.rand(4,1,device=DEVICE)*cfg.Lx; _y = torch.rand(4,1,device=DEVICE)*cfg.Ly
_t = torch.rand(4,1,device=DEVICE)*cfg.T
print("forward shape:", _net_dummy(_x, _y, _t).shape)
print("net params:", sum(p.numel() for p in _net_dummy.network_parameters()))


## 5. Loss Functions & Adaptive Weighting

The spatial diffusion field is now produced in a fully differentiable way
from the patient-specific WM mask: bilinear interpolation of the mask at
arbitrary `(x, y)`, then `D(x,y) = D_wm·m + D_gm·(1-m)`. This keeps
`∇·(D∇u)` exact under autograd.


In [ ]:
def lhs_collocation(cfg, N):
    s = lhs(3, N); s[:,0]*=cfg.Lx; s[:,1]*=cfg.Ly; s[:,2]*=cfg.T
    return torch.tensor(s, dtype=DTYPE, device=DEVICE)

def ic_points(cfg, N):
    s = lhs(2, N); s[:,0]*=cfg.Lx; s[:,1]*=cfg.Ly
    xy = torch.tensor(s, dtype=DTYPE, device=DEVICE)
    t0 = torch.zeros(N,1,dtype=DTYPE,device=DEVICE)
    return torch.cat([xy, t0], dim=-1)

def bc_points(cfg, N):
    side = np.random.randint(0,4,N); u01 = np.random.rand(N); t = np.random.rand(N)*cfg.T
    xs=np.zeros(N); ys=np.zeros(N); nx=np.zeros(N); ny=np.zeros(N)
    m=side==0; xs[m]=0;    ys[m]=u01[m]*cfg.Ly; nx[m]=-1
    m=side==1; xs[m]=cfg.Lx; ys[m]=u01[m]*cfg.Ly; nx[m]=+1
    m=side==2; ys[m]=0;    xs[m]=u01[m]*cfg.Lx; ny[m]=-1
    m=side==3; ys[m]=cfg.Ly; xs[m]=u01[m]*cfg.Lx; ny[m]=+1
    P = torch.tensor(np.stack([xs,ys,t],-1), dtype=DTYPE, device=DEVICE)
    Nh = torch.tensor(np.stack([nx,ny],-1), dtype=DTYPE, device=DEVICE)
    return P, Nh

def u0_fn_torch(cfg, x, y):
    return torch.exp(-((x-cfg.x0)**2 + (y-cfg.y0)**2) / (2*cfg.sigma0**2))

def _sample_mask(mask_tensor: torch.Tensor, x, y, cfg) -> torch.Tensor:
    """Differentiable bilinear sampling of a 2-D mask at physical (x,y)."""
    # Map (x,y) to grid_sample coords in [-1, 1]
    gx = 2.0 * x / cfg.Lx - 1.0
    gy = 2.0 * y / cfg.Ly - 1.0
    grid = torch.stack([gx.squeeze(-1), gy.squeeze(-1)], dim=-1).view(1, -1, 1, 2)
    inp = mask_tensor.view(1, 1, *mask_tensor.shape)
    out = F.grid_sample(inp, grid, mode="bilinear", padding_mode="border",
                        align_corners=True)
    return out.view(-1, 1)

def diffusion_field_torch(net, x, y, tau=1.5):
    D_wm, D_gm, _ = net.physical_params()
    if net.wm_mask.numel() > 0:
        m = _sample_mask(net.wm_mask, x, y, net.cfg)
        # Soften slightly for smoother gradients (no-op if already smooth)
        return D_wm * m + D_gm * (1.0 - m)
    # Analytic fallback (synthetic disc)
    gx, gy = net.cfg.gm_center
    r = torch.sqrt((x - gx)**2 + (y - gy)**2 + 1e-12)
    s = torch.sigmoid((r - net.cfg.gm_radius) / tau)
    return D_wm * s + D_gm * (1.0 - s)

def grad(y, x):
    return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y),
                               create_graph=True, retain_graph=True)[0]

def pde_residual(net, pts):
    x = pts[:,0:1].clone().requires_grad_(True)
    y = pts[:,1:2].clone().requires_grad_(True)
    t = pts[:,2:3].clone().requires_grad_(True)
    u = net(x, y, t)
    u_t = grad(u, t); u_x = grad(u, x); u_y = grad(u, y)
    u_xx = grad(u_x, x); u_yy = grad(u_y, y)
    D = diffusion_field_torch(net, x, y)
    D_x = grad(D, x); D_y = grad(D, y)
    _, _, rho = net.physical_params()
    div_D_grad = D_x*u_x + D_y*u_y + D*(u_xx + u_yy)
    return u_t - div_D_grad - rho * u * (1 - u)

def loss_pde(net, pts):  return (pde_residual(net, pts)**2).mean()
def loss_ic(net, pts):
    x=pts[:,0:1]; y=pts[:,1:2]; t=pts[:,2:3]
    return ((net(x,y,t) - u0_fn_torch(net.cfg,x,y))**2).mean()
def loss_bc(net, pts, normals):
    x=pts[:,0:1].clone().requires_grad_(True); y=pts[:,1:2].clone().requires_grad_(True); t=pts[:,2:3]
    u=net(x,y,t); u_x=grad(u,x); u_y=grad(u,y)
    return ((u_x*normals[:,0:1] + u_y*normals[:,1:2])**2).mean()
def loss_data(net, ox, oy, ot, ou): return ((net(ox,oy,ot) - ou)**2).mean()
def loss_reg(net, cfg):
    r = (net.log_D_wm - cfg.log_D_wm_prior_mean)**2 / (2*cfg.log_D_wm_prior_std**2)
    r = r + (net.log_D_gm - cfg.log_D_gm_prior_mean)**2 / (2*cfg.log_D_gm_prior_std**2)
    r = r + (net.log_rho  - cfg.log_rho_prior_mean)**2 / (2*cfg.log_rho_prior_std**2)
    return r.squeeze()
def loss_phys(net, pts):
    u = net(pts[:,0:1], pts[:,1:2], pts[:,2:3])
    return (F.relu(-u)**2 + F.relu(u-1)**2).mean()

def adapt_weights(net, loss_terms, old_lambdas, alpha):
    grad_norms = []
    params = net.network_parameters()
    for L in loss_terms:
        if not L.requires_grad: grad_norms.append(1e-12); continue
        gs = torch.autograd.grad(L, params, retain_graph=True, allow_unused=True)
        norms = [g.abs().max().item() for g in gs if g is not None]
        grad_norms.append(max(norms) if norms else 1e-12)
    total = sum(grad_norms) + 1e-30
    new = [alpha*old_lambdas[i] + (1-alpha)*(total/(gn+1e-30)) for i, gn in enumerate(grad_norms)]
    target_sum = float(sum(cfg.lambda_init))
    s = sum(new); new = [w*target_sum/s for w in new]
    return new

def rar_update(net, cfg, colloc):
    N = colloc.shape[0]
    n_rep = int(cfg.rar_fraction*N); n_cand = n_rep*cfg.rar_candidate_mult
    cand = lhs_collocation(cfg, n_cand)
    res = pde_residual(net, cand).detach().abs().squeeze()
    n_keep = max(n_rep, int(cfg.rar_top_fraction*n_cand))
    chosen = cand[torch.topk(res, n_keep).indices][:n_rep]
    idx = torch.randperm(N, device=colloc.device)[:n_rep]
    colloc = colloc.clone(); colloc[idx] = chosen
    return colloc

print("Losses ready.")


## 6. Training Loop

Three phases as before. The training set is the TCGA (or synthetic)
observations *minus* the 15% held-out split; the held-out points are used
for calibration/metrics in Sections 7–8.


In [ ]:
def to_tensor(a): return torch.tensor(np.asarray(a).reshape(-1,1), dtype=DTYPE, device=DEVICE)
OX  = to_tensor(obs_train["x"]); OY  = to_tensor(obs_train["y"])
OT  = to_tensor(obs_train["t"]); OU  = to_tensor(obs_train["u"])
OXv = to_tensor(obs_val["x"]);   OYv = to_tensor(obs_val["y"])
OTv = to_tensor(obs_val["t"]);   OUv = to_tensor(obs_val["u"])

def curriculum_T(cfg, it):
    if cfg.curriculum_every <= 0: return cfg.T
    steps = it // cfg.curriculum_every
    return cfg.T * min(1.0, cfg.curriculum_start_frac * (cfg.curriculum_growth**steps))

def rescale_time(pts, T_eff, T_full):
    pts = pts.clone(); pts[:,2:3] = pts[:,2:3] * (T_eff/T_full); return pts

def train_one_net(net, cfg, obs_tensors, verbose=True, tag=""):
    OX_,OY_,OT_,OU_ = obs_tensors
    lambdas = list(cfg.lambda_init)
    history = {k:[] for k in ["pde","ic","bc","data","reg","phys","total",
                              "D_wm","D_gm","rho","lambda"]}
    colloc = lhs_collocation(cfg, cfg.N_f)
    ic_pts = ic_points(cfg, cfg.N_ic)
    bcp, bcn = bc_points(cfg, cfg.N_bc)

    net.freeze_physics(True)
    opt = torch.optim.Adam(net.network_parameters(), lr=cfg.phase1_lr)
    pb = tqdm(range(cfg.phase1_iters), desc=f"{tag}Phase1", disable=not verbose)
    for it in pb:
        Te = curriculum_T(cfg, it)
        ce = rescale_time(colloc, Te, cfg.T); be = rescale_time(bcp, Te, cfg.T)
        Lp=loss_pde(net,ce); Li=loss_ic(net,ic_pts); Lb=loss_bc(net,be,bcn)
        Ld=loss_data(net,OX_,OY_,OT_,OU_); Lph=loss_phys(net,ce)
        total=lambdas[0]*Lp+lambdas[1]*Li+lambdas[2]*Lb+lambdas[3]*Ld+lambdas[5]*Lph
        opt.zero_grad(); total.backward()
        torch.nn.utils.clip_grad_norm_(net.network_parameters(), cfg.grad_clip_norm); opt.step()
        if it%cfg.log_every==0:
            Dw,Dg,r=[p.item() for p in net.physical_params()]
            history["pde"].append(Lp.item()); history["ic"].append(Li.item())
            history["bc"].append(Lb.item()); history["data"].append(Ld.item())
            history["reg"].append(0.0); history["phys"].append(Lph.item())
            history["total"].append(total.item())
            history["D_wm"].append(Dw); history["D_gm"].append(Dg); history["rho"].append(r)
            history["lambda"].append(list(lambdas))
            pb.set_postfix(pde=f"{Lp.item():.2e}", data=f"{Ld.item():.2e}")
        if it>0 and it%cfg.rar_every==0: colloc = rar_update(net, cfg, colloc)

    net.freeze_physics(False)
    all_params = net.network_parameters() + net.physics_parameters()
    opt = torch.optim.Adam([
        {"params": net.network_parameters(), "lr": cfg.phase2_lr},
        {"params": net.physics_parameters(), "lr": cfg.phase2_lr*2.0}])
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        opt, T_0=cfg.cosine_T0, T_mult=cfg.cosine_Tmult)
    pb = tqdm(range(cfg.phase2_iters), desc=f"{tag}Phase2", disable=not verbose)
    for it in pb:
        Te = curriculum_T(cfg, it)
        ce = rescale_time(colloc, Te, cfg.T); be = rescale_time(bcp, Te, cfg.T)
        Lp=loss_pde(net,ce); Li=loss_ic(net,ic_pts); Lb=loss_bc(net,be,bcn)
        Ld=loss_data(net,OX_,OY_,OT_,OU_); Lr=loss_reg(net,cfg); Lph=loss_phys(net,ce)
        total=(lambdas[0]*Lp+lambdas[1]*Li+lambdas[2]*Lb+lambdas[3]*Ld+lambdas[4]*Lr+lambdas[5]*Lph)
        opt.zero_grad(); total.backward()
        torch.nn.utils.clip_grad_norm_(all_params, cfg.grad_clip_norm); opt.step(); sched.step()
        if it%cfg.log_every==0:
            Dw,Dg,r=[p.item() for p in net.physical_params()]
            history["pde"].append(Lp.item()); history["ic"].append(Li.item())
            history["bc"].append(Lb.item()); history["data"].append(Ld.item())
            history["reg"].append(Lr.item()); history["phys"].append(Lph.item())
            history["total"].append(total.item())
            history["D_wm"].append(Dw); history["D_gm"].append(Dg); history["rho"].append(r)
            history["lambda"].append(list(lambdas))
            pb.set_postfix(pde=f"{Lp.item():.2e}", data=f"{Ld.item():.2e}",
                           D_wm=f"{Dw:.3f}", rho=f"{r:.4f}")
        if it>0 and it%cfg.adaptive_every==0:
            terms=[loss_pde(net,ce),loss_ic(net,ic_pts),loss_bc(net,be,bcn),
                   loss_data(net,OX_,OY_,OT_,OU_),loss_reg(net,cfg),loss_phys(net,ce)]
            lambdas = adapt_weights(net, terms, lambdas, cfg.adaptive_alpha)
        if it>0 and it%cfg.rar_every==0: colloc = rar_update(net, cfg, colloc)

    lbfgs = torch.optim.LBFGS(all_params, lr=1.0, max_iter=20,
                              history_size=50, line_search_fn="strong_wolfe")
    pb = tqdm(range(cfg.phase3_iters//20), desc=f"{tag}Phase3", disable=not verbose)
    for it in pb:
        def closure():
            lbfgs.zero_grad()
            Lp=loss_pde(net,colloc); Li=loss_ic(net,ic_pts); Lb=loss_bc(net,bcp,bcn)
            Ld=loss_data(net,OX_,OY_,OT_,OU_); Lr=loss_reg(net,cfg); Lph=loss_phys(net,colloc)
            tot=(lambdas[0]*Lp+lambdas[1]*Li+lambdas[2]*Lb+lambdas[3]*Ld+lambdas[4]*Lr+lambdas[5]*Lph)
            tot.backward(); return tot
        loss=lbfgs.step(closure); pb.set_postfix(total=f"{loss.item():.3e}")
    return history, lambdas, (colloc, ic_pts, bcp, bcn)


print("Training primary PINN...")
pinn = TumorPINN(cfg, wm_mask_np=wm_full).to(DEVICE)
history, final_lambdas, sampling_state = train_one_net(
    pinn, cfg, (OX,OY,OT,OU), tag="[primary] ")
D_wm_est, D_gm_est, rho_est = [p.item() for p in pinn.physical_params()]
print(f"\nRecovered: D_wm={D_wm_est:.4f}  D_gm={D_gm_est:.4f}  rho={rho_est:.5f}")
if not USE_REAL_DATA:
    print(f"True     : D_wm={cfg.D_wm_true:.4f}  D_gm={cfg.D_gm_true:.4f}  rho={cfg.rho_true:.5f}")

print(f"\nTraining deep ensemble (K={cfg.ensemble_size})...")
ensemble = []
for k in range(cfg.ensemble_size):
    torch.manual_seed(SEED + 37*k); np.random.seed(SEED + 37*k)
    m = TumorPINN(cfg, wm_mask_np=wm_full, seed=SEED + 37*k).to(DEVICE)
    _,_,_ = train_one_net(m, cfg, (OX,OY,OT,OU), verbose=False, tag=f"[ens{k}] ")
    ensemble.append(m)
    Dw,Dg,r = [p.item() for p in m.physical_params()]
    print(f"  ens[{k}]: D_wm={Dw:.4f}  D_gm={Dg:.4f}  rho={r:.5f}")


## 7. Uncertainty Quantification

Same three methods — MC Dropout, deep ensemble, Laplace over physical
parameters. The Laplace Hessian is evaluated w.r.t. `[log_D_wm, log_D_gm,
log_ρ]` at the MAP point.


In [ ]:
def eval_grid_tensors(cfg, t_val, n=None):
    n = n or cfg.eval_grid
    xs = np.linspace(0, cfg.Lx, n); ys = np.linspace(0, cfg.Ly, n)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    fx = torch.tensor(X.reshape(-1,1), dtype=DTYPE, device=DEVICE)
    fy = torch.tensor(Y.reshape(-1,1), dtype=DTYPE, device=DEVICE)
    ft = torch.full_like(fx, t_val)
    return X, Y, fx, fy, ft

def mc_dropout_predict(net, fx, fy, ft, M):
    net.train()
    preds=[]
    with torch.no_grad():
        for _ in range(M): preds.append(net(fx, fy, ft).cpu().numpy().squeeze())
    net.eval()
    preds = np.stack(preds, 0)
    return preds.mean(0), preds.std(0)

def ensemble_predict(models, fx, fy, ft):
    mus, vars_ = [], []
    for m in models:
        mu, sd = mc_dropout_predict(m, fx, fy, ft, M=50)
        mus.append(mu); vars_.append(sd**2)
    mus = np.stack(mus,0); vars_ = np.stack(vars_,0)
    mu_ens = mus.mean(0)
    var_ens = (vars_ + mus**2).mean(0) - mu_ens**2
    return mu_ens, np.sqrt(np.clip(var_ens, 0, None))

def total_loss_fn(net, colloc, ic_pts, bcp, bcn, obs_t, lambdas):
    OX_,OY_,OT_,OU_ = obs_t
    return (lambdas[0]*loss_pde(net,colloc) + lambdas[1]*loss_ic(net,ic_pts) +
            lambdas[2]*loss_bc(net,bcp,bcn) + lambdas[3]*loss_data(net,OX_,OY_,OT_,OU_) +
            lambdas[4]*loss_reg(net,cfg) + lambdas[5]*loss_phys(net,colloc))

def laplace_posterior(net, sampling_state, lambdas, obs_t):
    colloc, ic_pts, bcp, bcn = sampling_state
    def _loss(a, b, c):
        saved=(net.log_D_wm.data.clone(),net.log_D_gm.data.clone(),net.log_rho.data.clone())
        net.log_D_wm.data.copy_(a); net.log_D_gm.data.copy_(b); net.log_rho.data.copy_(c)
        L = total_loss_fn(net, colloc, ic_pts, bcp, bcn, obs_t, lambdas)
        net.log_D_wm.data.copy_(saved[0]); net.log_D_gm.data.copy_(saved[1]); net.log_rho.data.copy_(saved[2])
        return L
    p0 = (net.log_D_wm.detach().clone(), net.log_D_gm.detach().clone(), net.log_rho.detach().clone())
    try:
        from torch.autograd.functional import hessian
        H = hessian(_loss, p0, create_graph=False, strict=False)
        Hm = torch.zeros(3,3)
        for i in range(3):
            for j in range(3): Hm[i,j] = H[i][j].reshape(-1)[0]
    except Exception as e:
        print(f"[Laplace] full hessian failed ({e}); diagonal Fisher fallback.")
        Hm = torch.zeros(3,3); params=[net.log_D_wm,net.log_D_gm,net.log_rho]
        for _ in range(20):
            L = total_loss_fn(net, colloc, ic_pts, bcp, bcn, obs_t, lambdas)
            gs = torch.autograd.grad(L, params, retain_graph=False, allow_unused=True)
            for i, g in enumerate(gs):
                if g is not None: Hm[i,i] += g.item()**2
        Hm /= 20.0
    Hm = Hm + 1e-4 * torch.eye(3)
    Sigma = torch.linalg.inv(Hm).cpu()
    eigs = torch.linalg.eigvalsh(Sigma)
    if eigs.min() < 0: Sigma = Sigma - (eigs.min() - 1e-6) * torch.eye(3)
    mean = torch.tensor([p0[0].item(), p0[1].item(), p0[2].item()])
    dist = torch.distributions.MultivariateNormal(mean, covariance_matrix=Sigma)
    return dict(mean=mean.numpy(), cov=Sigma.numpy(),
                samples=torch.exp(dist.sample((cfg.laplace_samples,))).numpy())

X_eval, Y_eval, fx_eval, fy_eval, ft_eval = eval_grid_tensors(cfg, cfg.T_pred)
print("MC-Dropout...");  u_mcd_mean, u_mcd_std = mc_dropout_predict(pinn, fx_eval, fy_eval, ft_eval, cfg.mcd_samples)
print("Ensemble...");    u_ens_mean, u_ens_std = ensemble_predict(ensemble, fx_eval, fy_eval, ft_eval)
print("Laplace...");     laplace = laplace_posterior(pinn, sampling_state, final_lambdas, (OX,OY,OT,OU))
S = laplace["samples"]
for i,name in enumerate(["D_wm","D_gm","rho"]):
    print(f"  {name:>5s} CI95: [{np.quantile(S[:,i],0.025):.5f}, {np.quantile(S[:,i],0.975):.5f}]")


## 8. Evaluation & Metrics

**TCGA mode**: there is no ground-truth D/ρ. We evaluate:
* Held-out observation MSE and R²
* L2 relative error of PINN mean vs the tumor density proxy at
  `T_pred` (the latest session)
* DICE, Hausdorff on the tumor mask (u > 0.16)
* PDE residual, coverage, sharpness

**Synthetic mode** additionally reports parameter recovery errors.


In [ ]:
from scipy.ndimage import binary_erosion

def dice(a, b):
    a=a.astype(bool); b=b.astype(bool)
    if a.sum()+b.sum()==0: return 1.0
    return 2*(a&b).sum() / (a.sum()+b.sum())

def contour_points(mask, xs, ys):
    boundary = mask & ~binary_erosion(mask)
    jj, ii = np.where(boundary)
    return np.zeros((0,2)) if len(ii)==0 else np.stack([xs[ii], ys[jj]], -1)

def hausdorff(A, B):
    if len(A)==0 or len(B)==0: return float("nan")
    from scipy.spatial import cKDTree
    return max(cKDTree(B).query(A)[0].max(), cKDTree(A).query(B)[0].max())

U_true_grid = _bilinear(TRUE_U_GRID_X, TRUE_U_GRID_Y, TRUE_U_AT_T_PRED,
                        X_eval.reshape(-1), Y_eval.reshape(-1)).reshape(X_eval.shape)
u_pinn_mean = u_ens_mean.reshape(X_eval.shape)
u_pinn_std  = u_ens_std.reshape(X_eval.shape)

# Held-out observation fit
with torch.no_grad():
    u_val_pred = pinn(OXv, OYv, OTv).cpu().numpy().squeeze()
val_mse = float(np.mean((u_val_pred - obs_val["u"])**2))
ss_res = np.sum((obs_val["u"] - u_val_pred)**2)
ss_tot = np.sum((obs_val["u"] - obs_val["u"].mean())**2) + 1e-12
val_r2 = float(1.0 - ss_res/ss_tot)

l2_rel = float(np.linalg.norm(u_pinn_mean - U_true_grid) / (np.linalg.norm(U_true_grid) + 1e-12))
m_true = U_true_grid > cfg.dice_threshold
m_pinn = u_pinn_mean > cfg.dice_threshold
dice_score = dice(m_pinn, m_true)
pts_true = contour_points(m_true, X_eval[0], Y_eval[:,0])
pts_pinn = contour_points(m_pinn, X_eval[0], Y_eval[:,0])
hd = hausdorff(pts_true, pts_pinn)

with torch.enable_grad():
    dense = lhs_collocation(cfg, 5000)
    mean_res = pde_residual(pinn, dense).detach().abs().mean().item()

ci_lo = u_pinn_mean - 1.96*u_pinn_std; ci_hi = u_pinn_mean + 1.96*u_pinn_std
coverage = float(np.mean((U_true_grid >= ci_lo) & (U_true_grid <= ci_hi)))
sharpness = float((ci_hi - ci_lo).mean())

metrics = dict(val_mse=val_mse, val_r2=val_r2,
               l2_rel=l2_rel, dice=dice_score, hausdorff=hd,
               pde_residual_mean=mean_res, coverage=coverage, sharpness=sharpness)
if not USE_REAL_DATA:
    metrics["eps_Dwm"] = 100*abs(D_wm_est - cfg.D_wm_true)/cfg.D_wm_true
    metrics["eps_Dgm"] = 100*abs(D_gm_est - cfg.D_gm_true)/cfg.D_gm_true
    metrics["eps_rho"] = 100*abs(rho_est  - cfg.rho_true) /cfg.rho_true

for k,v in metrics.items(): print(f"{k:>20s}: {v:.4f}")


## 9. Visualization

In [ ]:
# --- Fig 1: training curves & parameter traces ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
for k in ["pde","ic","bc","data","reg","phys","total"]:
    if len(history[k])==0: continue
    ax.plot(history[k], label=k, linewidth=1.2)
ax.set_yscale("log"); ax.set_xlabel("Log step"); ax.set_ylabel("Loss")
ax.set_title("Loss components"); ax.legend(ncol=2, fontsize=8)
ax = axes[1]
ax.plot(history["D_wm"], label="D_wm", color="C0")
ax.plot(history["D_gm"], label="D_gm", color="C2")
if not USE_REAL_DATA:
    ax.axhline(cfg.D_wm_true, color="C0", linestyle="--", alpha=0.6)
    ax.axhline(cfg.D_gm_true, color="C2", linestyle="--", alpha=0.6)
ax.set_ylabel("D [mm²/day]"); ax.set_xlabel("Log step")
ax2 = ax.twinx(); ax2.plot(history["rho"], color="C3", label="ρ")
if not USE_REAL_DATA: ax2.axhline(cfg.rho_true, color="C3", linestyle="--", alpha=0.6)
ax2.set_ylabel("ρ [1/day]", color="C3"); ax.legend(loc="upper left", fontsize=8)
ax.set_title("Physical parameter estimates")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"01_training.png", dpi=300); plt.show()


# --- Fig 2: snapshot comparison at each observation time ---
obs_ts_unique = sorted(set(obs["t"].tolist()))
fig, axes = plt.subplots(3, len(obs_ts_unique), figsize=(5*len(obs_ts_unique), 12))
if len(obs_ts_unique)==1: axes = axes[:,None]
for j, ts in enumerate(obs_ts_unique):
    if USE_REAL_DATA:
        # nearest frame
        f = min(frames, key=lambda f: abs(f["day"]-ts))
        U_true = f["density"]
        xg = np.linspace(0, cfg.Lx, U_true.shape[1]); yg = np.linspace(0, cfg.Ly, U_true.shape[0])
        U_true_re = _bilinear(xg, yg, U_true, X_eval.reshape(-1), Y_eval.reshape(-1)).reshape(X_eval.shape)
    else:
        key = min(fd["snaps"].keys(), key=lambda k: abs(k-ts))
        U_true_re = _bilinear(fd["x"], fd["y"], fd["snaps"][key],
                              X_eval.reshape(-1), Y_eval.reshape(-1)).reshape(X_eval.shape)
    _,_,fx_,fy_,ft_ = eval_grid_tensors(cfg, ts)
    mu, sd = mc_dropout_predict(pinn, fx_, fy_, ft_, M=60)
    mu = mu.reshape(cfg.eval_grid, cfg.eval_grid); sd = sd.reshape(cfg.eval_grid, cfg.eval_grid)
    im0=axes[0,j].imshow(U_true_re, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="viridis", vmin=0, vmax=1)
    axes[0,j].set_title(("obs density" if USE_REAL_DATA else "FD truth")+f"  t={ts:.0f}d")
    im1=axes[1,j].imshow(mu, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="viridis", vmin=0, vmax=1)
    axes[1,j].set_title(f"PINN mean  t={ts:.0f}d")
    im2=axes[2,j].imshow(sd, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="magma")
    axes[2,j].set_title(f"PINN std  t={ts:.0f}d")
    for im, ax in zip([im0,im1,im2], axes[:,j]):
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.savefig(RESULTS_DIR/"02_snapshot_comparison.png", dpi=300); plt.show()


# --- Fig 3: forward prediction + MRI overlay at T_pred ---
fig, axes = plt.subplots(1, 4 if USE_REAL_DATA else 3, figsize=(5*4, 5))
im=axes[0].imshow(U_true_grid, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="viridis", vmin=0, vmax=1)
axes[0].set_title(f"{'Obs density' if USE_REAL_DATA else 'FD truth'} @ T_pred={cfg.T_pred:.0f}d"); plt.colorbar(im, ax=axes[0])
im=axes[1].imshow(u_pinn_mean, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="viridis", vmin=0, vmax=1)
axes[1].set_title("PINN ensemble mean"); plt.colorbar(im, ax=axes[1])
im=axes[2].imshow(u_pinn_std, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="magma")
axes[2].set_title("Ensemble std"); plt.colorbar(im, ax=axes[2])
if USE_REAL_DATA:
    ax = axes[3]
    t1_bg = frames[-1]["t1ce"] if frames[-1].get("t1ce") is not None else frames[-1]["t1"]
    ax.imshow(t1_bg, origin="lower", extent=[0,cfg.Lx,0,cfg.Ly], cmap="gray")
    ax.contour(X_eval, Y_eval, u_pinn_mean, levels=[0.16, 0.5, 0.8], colors=["yellow","orange","red"])
    ax.set_title("PINN iso-contours on T1ce")
txt = f"D_wm: {D_wm_est:.3f} | D_gm: {D_gm_est:.3f} | ρ: {rho_est:.4f}"
if not USE_REAL_DATA:
    txt = (f"D_wm: {cfg.D_wm_true:.3f}->{D_wm_est:.3f} ({metrics['eps_Dwm']:.1f}% err)   "
           f"D_gm: {cfg.D_gm_true:.3f}->{D_gm_est:.3f} ({metrics['eps_Dgm']:.1f}% err)   "
           f"ρ: {cfg.rho_true:.4f}->{rho_est:.4f} ({metrics['eps_rho']:.1f}% err)")
fig.suptitle(txt, fontsize=10)
plt.tight_layout(); plt.savefig(RESULTS_DIR/"03_forward_prediction.png", dpi=300); plt.show()


# --- Fig 4: Laplace posterior (D_wm, rho) ---
from matplotlib.gridspec import GridSpec
fig = plt.figure(figsize=(7,7)); gs = GridSpec(4,4,figure=fig)
axm = fig.add_subplot(gs[1:,:3]); axt = fig.add_subplot(gs[0,:3], sharex=axm)
axr = fig.add_subplot(gs[1:,3], sharey=axm)
axm.scatter(S[:,0], S[:,2], s=5, alpha=0.3, color="C0")
if not USE_REAL_DATA:
    axm.scatter([cfg.D_wm_true],[cfg.rho_true], marker="*", color="red", s=200, label="true"); axm.legend()
axm.scatter([D_wm_est],[rho_est], marker="x", color="black", s=120, label="MAP")
axm.set_xlabel("D_wm [mm²/day]"); axm.set_ylabel("ρ [1/day]"); axm.legend()
axt.hist(S[:,0], bins=40, color="C0")
axt.axvspan(np.quantile(S[:,0],0.025), np.quantile(S[:,0],0.975), color="C0", alpha=0.2)
axr.hist(S[:,2], bins=40, orientation="horizontal", color="C0")
axr.axhspan(np.quantile(S[:,2],0.025), np.quantile(S[:,2],0.975), color="C0", alpha=0.2)
if not USE_REAL_DATA:
    axt.axvline(cfg.D_wm_true, color="red"); axr.axhline(cfg.rho_true, color="red")
plt.suptitle("Laplace posterior over physical parameters")
plt.tight_layout(); plt.savefig(RESULTS_DIR/"04_laplace_posterior.png", dpi=300); plt.show()


# --- Fig 5: 1-D slice through tumor center at T_pred ---
j_mid = cfg.eval_grid // 2
xs = X_eval[j_mid,:]
line_true = U_true_grid[j_mid,:]
mcd_mu = u_mcd_mean.reshape(cfg.eval_grid, cfg.eval_grid)[j_mid,:]
mcd_sd = u_mcd_std.reshape(cfg.eval_grid, cfg.eval_grid)[j_mid,:]
ens_mu = u_pinn_mean[j_mid,:]; ens_sd = u_pinn_std[j_mid,:]
n_sub = min(200, cfg.laplace_samples)
idx_sub = np.random.choice(S.shape[0], n_sub, replace=False)
saved=(pinn.log_D_wm.data.clone(),pinn.log_D_gm.data.clone(),pinn.log_rho.data.clone())
pinn.eval(); lap_preds=[]
with torch.no_grad():
    for s in S[idx_sub]:
        pinn.log_D_wm.data = torch.tensor(math.log(max(s[0],1e-8)), dtype=DTYPE, device=DEVICE)
        pinn.log_D_gm.data = torch.tensor(math.log(max(s[1],1e-8)), dtype=DTYPE, device=DEVICE)
        pinn.log_rho.data  = torch.tensor(math.log(max(s[2],1e-8)), dtype=DTYPE, device=DEVICE)
        fx_l = torch.tensor(xs.reshape(-1,1), dtype=DTYPE, device=DEVICE)
        fy_l = torch.full_like(fx_l, float(Y_eval[j_mid,0]))
        ft_l = torch.full_like(fx_l, cfg.T_pred)
        lap_preds.append(pinn(fx_l, fy_l, ft_l).cpu().numpy().squeeze())
pinn.log_D_wm.data.copy_(saved[0]); pinn.log_D_gm.data.copy_(saved[1]); pinn.log_rho.data.copy_(saved[2])
lap_preds = np.stack(lap_preds); lap_mu = lap_preds.mean(0); lap_sd = lap_preds.std(0)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(xs, line_true, "k-", lw=2, label="Observed density" if USE_REAL_DATA else "FD truth")
for mu, sd, c, n in [(mcd_mu,mcd_sd,"C0","MC Dropout"),
                     (ens_mu,ens_sd,"C1","Deep Ensemble"),
                     (lap_mu,lap_sd,"C2","Laplace")]:
    ax.plot(xs, mu, color=c, label=f"{n} mean")
    ax.fill_between(xs, mu-2*sd, mu+2*sd, color=c, alpha=0.2)
ax.set_xlabel("x [mm]"); ax.set_ylabel("u")
ax.set_title(f"1-D slice @ y={Y_eval[j_mid,0]:.0f} mm, t={cfg.T_pred:.0f}d")
ax.legend(); plt.tight_layout()
plt.savefig(RESULTS_DIR/"05_slice_uq.png", dpi=300); plt.show()


# --- Fig 6: calibration reliability diagram ---
from scipy.stats import norm
levels = np.linspace(0.05, 0.95, 19); observed=[]
for p in levels:
    zp = norm.ppf(0.5 + p/2)
    lo = u_pinn_mean - zp*u_pinn_std; hi = u_pinn_mean + zp*u_pinn_std
    observed.append(float(np.mean((U_true_grid >= lo) & (U_true_grid <= hi))))
fig, ax = plt.subplots(figsize=(6,6))
ax.plot([0,1],[0,1],"k--", label="ideal")
ax.plot(levels, observed, "o-", color="C0", label="ensemble")
ax.set_xlabel("Expected coverage"); ax.set_ylabel("Observed coverage")
ax.set_title("Calibration reliability diagram"); ax.legend()
plt.tight_layout(); plt.savefig(RESULTS_DIR/"06_calibration.png", dpi=300); plt.show()


# --- Fig 7 (TCGA only): MRI + WM mask + tumor density at all sessions ---
if USE_REAL_DATA:
    fig, axes = plt.subplots(len(frames), 4, figsize=(16, 4*len(frames)))
    if len(frames) == 1: axes = axes[None,:]
    for i, f in enumerate(frames):
        axes[i,0].imshow(f["t1"], cmap="gray", origin="lower"); axes[i,0].set_title(f"T1 (day {f['day']:.0f})")
        axes[i,1].imshow(f["t1ce"] if f["t1ce"] is not None else f["t1"], cmap="gray", origin="lower")
        axes[i,1].set_title("T1ce")
        axes[i,2].imshow(f["wm"], cmap="Greys", origin="lower"); axes[i,2].set_title("Derived WM mask")
        axes[i,3].imshow(f["density"], cmap="viridis", origin="lower", vmin=0, vmax=1); axes[i,3].set_title("Tumor density proxy")
        for a in axes[i]: a.axis("off")
    plt.suptitle(f"TCGA-GBM case: {frames[0]['case']}  (slice {frames[0]['slice_index']})", y=1.02)
    plt.tight_layout(); plt.savefig(RESULTS_DIR/"07_tcga_inputs.png", dpi=300); plt.show()


## 10. Results Summary

In [ ]:
def fmt_ci(samples):
    return f"[{np.quantile(samples,0.025):.5f}, {np.quantile(samples,0.975):.5f}]"

print("\n" + "="*80)
print(f"  {'TCGA-GBM CASE' if USE_REAL_DATA else 'SYNTHETIC'} — PARAMETER ESTIMATES")
print("="*80)
if USE_REAL_DATA:
    print(f"Case: {frames[0]['case']}   Slice index: {frames[0]['slice_index']}   "
          f"Sessions (days): {[round(f['day'],1) for f in frames]}")
    print(f"\n{'param':<16} {'est':>12} {'95% CI (Laplace)':>30}")
    print(f"{'D_wm [mm²/day]':<16} {D_wm_est:>12.5f} {fmt_ci(S[:,0]):>30}")
    print(f"{'D_gm [mm²/day]':<16} {D_gm_est:>12.5f} {fmt_ci(S[:,1]):>30}")
    print(f"{'rho  [1/day]':<16} {rho_est:>12.5f} {fmt_ci(S[:,2]):>30}")
else:
    rows = [("D_wm", cfg.D_wm_true, D_wm_est, metrics["eps_Dwm"], fmt_ci(S[:,0])),
            ("D_gm", cfg.D_gm_true, D_gm_est, metrics["eps_Dgm"], fmt_ci(S[:,1])),
            ("rho",  cfg.rho_true,  rho_est,  metrics["eps_rho"], fmt_ci(S[:,2]))]
    print(f"{'param':<8} {'true':>12} {'est':>12} {'err %':>8} {'95% CI':>30}")
    for n,t,e,err,ci in rows:
        print(f"{n:<8} {t:>12.5f} {e:>12.5f} {err:>8.2f} {ci:>30}")

print("\n" + "="*80)
print(f"  HELD-OUT OBSERVATION FIT")
print("="*80)
print(f"{'Validation MSE':<30}: {metrics['val_mse']:.5f}")
print(f"{'Validation R²':<30}: {metrics['val_r2']:.4f}")

print("\n" + "="*80)
print(f"  FIELD METRICS @ T_pred = {cfg.T_pred:.0f}d")
print("="*80)
print(f"{'L2 relative error':<30}: {metrics['l2_rel']:.4f}")
print(f"{'DICE @ τ=0.16':<30}: {metrics['dice']:.4f}")
print(f"{'Hausdorff [mm]':<30}: {metrics['hausdorff']:.4f}")
print(f"{'Mean |PDE residual|':<30}: {metrics['pde_residual_mean']:.4e}")

print("\n" + "="*80)
print("  UQ — CALIBRATION")
print("="*80)
print(f"{'95% coverage':<30}: {metrics['coverage']*100:.1f}%")
print(f"{'95% CI mean width':<30}: {metrics['sharpness']:.4f}")

sharp_mcd = (1.96 * u_mcd_std).mean() * 2
sharp_ens = metrics["sharpness"]
sharp_lap = float((1.96 * lap_sd).mean() * 2)
tightest = min([("MC Dropout", sharp_mcd), ("Ensemble", sharp_ens), ("Laplace", sharp_lap)],
               key=lambda p: p[1])
print(f"\nTightest mean 95% CI: {tightest[0]} ({tightest[1]:.3f})")
if USE_REAL_DATA:
    within_lit = (0.05 <= D_wm_est <= 1.5) and (0.005 <= D_gm_est <= 0.3) and (0.001 <= rho_est <= 0.1)
    print(f"Estimates within literature ranges: {'YES' if within_lit else 'NO — review case / data quality'}")
    if len(frames) == 1:
        print("NOTE: single-session case — D and ρ are weakly identifiable. "
              "Interpret CI widths accordingly; prefer multi-session data.")
else:
    print("Parameter recovery:",
          "accurate" if (metrics["eps_Dwm"]<30 and metrics["eps_rho"]<30) else "approximate")
print("\nFigures saved to ./results/ at 300 DPI.")
